# Chapter 1: Common Conventions and API Elements of scikit-learn

*Reproduction + theoretical summary of Chapter 1 of **scikit-learn Cookbook, Third Edition** (John Sukup, O'Reilly/Packt).*

Before any specific algorithm is introduced, the book spends its first chapter on something more basic: **the shared "grammar" that every scikit-learn object speaks**. Once that grammar is clear, a random forest, a scaler and a clustering model all feel like variations of the same tool.

This notebook walks through the nine recipes of the chapter in order. For each one you will find (1) a short theory section, (2) runnable code reproducing the book's example, and (3) a few small extra experiments that make the idea easier to see.

| # | Recipe | Core idea |
|---|--------|-----------|
| 1 | Design philosophy | consistency, simplicity, modularity, reusability |
| 2 | Estimators | `fit()`, `predict()`, `fit_predict()` |
| 3 | Transformers | `fit()`, `transform()`, `fit_transform()` |
| 4 | Custom estimators / transformers | `BaseEstimator` + mixins |
| 5 | Pipelines | chain steps into one object (and a note on MLOps) |
| 6 | Common attributes and methods | `coef_`, `intercept_`, `score()` |
| 7 | Hyperparameter tuning | `set_params()`, `get_params()`, grid / random / halving search |
| 8 | Metadata | estimator tags and metadata routing |
| 9 | Best practices | a checklist for day-to-day API use |

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import sklearn

print("scikit-learn:", sklearn.__version__)
print("NumPy       :", np.__version__)

scikit-learn: 1.9.1
NumPy       : 2.5.3


## 1. Design philosophy of scikit-learn

The library is held together by four ideas:

| Principle | Meaning in practice |
|-----------|--------------------|
| **Consistency** | Almost every object exposes the same small set of verbs: `fit`, `predict`, `transform`. Learn them once, use them everywhere. |
| **Simplicity** | Sensible default hyperparameters mean a first working model takes only a few lines. Tuning comes later. |
| **Modularity** | Estimators, transformers and pipelines are independent blocks that can be recombined freely. |
| **Reusability** | Anything that follows the conventions (including *your own* classes) plugs into the rest of the ecosystem unchanged. |

The practical payoff is that **swapping one algorithm for another is a one-line change**. The cell below trains four very different classifiers on the Iris dataset with the exact same loop body.

*A small spelling note from the book: the name is always written in lowercase, **scikit-learn**, and pronounced "sy-kit learn", where "sci" stands for science.*

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

X_iris, y_iris = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_iris, y_iris, test_size=0.25, random_state=42, stratify=y_iris
)

candidates = [
    LogisticRegression(max_iter=1000),
    KNeighborsClassifier(),
    DecisionTreeClassifier(random_state=0),
    RandomForestClassifier(random_state=0),
]

for model in candidates:                 # <- the loop body never changes
    model.fit(X_tr, y_tr)
    print(f"{type(model).__name__:<24} test accuracy = {model.score(X_te, y_te):.4f}")

LogisticRegression       test accuracy = 0.9474
KNeighborsClassifier     test accuracy = 0.9737
DecisionTreeClassifier   test accuracy = 0.9211
RandomForestClassifier   test accuracy = 0.8947


The loop body is identical for all four models. Only the *list* changes. (With just 38 test samples these accuracies differ by one or two samples, so they should not be read as a ranking of the algorithms. The point here is the interface, not the leaderboard.)

## 2. Understanding estimators

### 2.1 What is an estimator?

An **estimator** is any Python object that *learns something from data*. This covers classifiers, regressors, clustering models and also preprocessing tools. Every estimator implements `fit()`.

For supervised learning we have a feature matrix $\mathbf{X}\in\mathbb{R}^{n\times p}$ ($n$ samples, $p$ features) and a target vector $\mathbf{y}$. `fit(X, y)` searches for a function $\hat f$ such that $\hat f(\mathbf{X})\approx \mathbf{y}$, and `predict(X_new)` evaluates that function on unseen rows.

The usual three-step routine is therefore:

```
model = Estimator(hyperparameters...)   # 1. configure (nothing is learned yet)
model.fit(X_train, y_train)             # 2. learn
model.predict(X_new)                    # 3. use
```

### 2.2 Example: `LinearRegression`

Linear regression chooses weights $\mathbf{w}$ and an intercept $b$ that minimise the sum of squared residuals:

$$\min_{\mathbf{w},b}\ \sum_{i=1}^{n}\bigl(y_i-(\mathbf{w}^{\top}\mathbf{x}_i+b)\bigr)^2 ,\qquad
\hat{\mathbf{w}}=(\mathbf{X}^{\top}\mathbf{X})^{-1}\mathbf{X}^{\top}\mathbf{y}\ \text{(closed form, intercept absorbed)}$$

In [3]:
from sklearn.linear_model import LinearRegression

X = np.array([[1], [2], [3], [4], [5]])      # 5 samples, 1 feature
y = np.array([1, 2, 3, 3.5, 5])              # target values

model = LinearRegression()
model.fit(X, y)

X_new = np.array([[6], [7]])
print("Predictions for x = 6 and x = 7:", model.predict(X_new))

Predictions for x = 6 and x = 7: [5.75 6.7 ]


The model returns **5.75** and **6.70**. We can double-check by hand using the learned line $\hat y = 0.95\,x + 0.05$: for $x=6$ this gives $0.95\times 6+0.05=5.75$ ✓. (The learned numbers are inspected properly in Section 6.)

### 2.3 `fit_predict()` and when it makes sense

`fit_predict(X)` is simply `fit(X)` followed by `predict(X)` on the *same* data. It is mainly used for **unsupervised** algorithms such as clustering, where there is no $\mathbf{y}$ to learn from and what we want is a label for each training point.

K-Means partitions the data into $K$ groups by minimising the within-cluster sum of squares

$$\min_{C_1,\dots,C_K}\ \sum_{k=1}^{K}\ \sum_{\mathbf{x}_i\in C_k}\lVert \mathbf{x}_i-\boldsymbol{\mu}_k\rVert^{2},\qquad \boldsymbol{\mu}_k=\frac{1}{|C_k|}\sum_{\mathbf{x}_i\in C_k}\mathbf{x}_i .$$

In [4]:
from sklearn.cluster import KMeans

X = np.array([[1], [2], [3], [4], [5]])

kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
labels = kmeans.fit_predict(X)

print("Cluster labels :", labels.tolist())
print("Cluster centres:", kmeans.cluster_centers_.ravel())

Cluster labels : [0, 0, 0, 1, 1]
Cluster centres: [2.  4.5]


The points $\{1,2,3\}$ land in one cluster (centre 2.0) and $\{4,5\}$ in the other (centre 4.5). Note that the numeric label (0 or 1) is arbitrary: another random seed could swap them without changing the grouping.

**Rule of thumb**

| Situation | Typical call |
|-----------|--------------|
| Supervised learning | `fit(X_train, y_train)`, then `predict(X_test)` on *held-out* data |
| Clustering / unsupervised, labels for the training data itself | `fit_predict(X)` |
| Unsupervised but you want to check generalisation | split the data and use `fit` + `predict` separately, exactly like supervised work |

## 3. Transformers and the `transform()` method

A **transformer** is an estimator whose output is *new data* rather than predictions. It follows the same `fit()` convention but adds:

| Method | What it does |
|--------|--------------|
| `fit(X)` | learn parameters from data (e.g. column means and standard deviations) |
| `transform(X)` | apply the learned parameters |
| `fit_transform(X)` | both steps in one call |

### 3.1 `StandardScaler` (z-score standardisation)

$$z_{ij}=\frac{x_{ij}-\mu_j}{\sigma_j}$$

where $\mu_j$ and $\sigma_j$ are the mean and the (population) standard deviation of column $j$. After scaling, each column has mean 0 and standard deviation 1.

*Why bother?* Distance-based and gradient-based methods (k-NN, SVM, logistic regression with regularisation, neural networks) are sensitive to the scale of the inputs. A feature measured in the thousands would otherwise drown out one measured in fractions.

In [5]:
from sklearn.preprocessing import StandardScaler

X = np.array([[1, 2],
              [3, 4],
              [5, 6]])

scaler = StandardScaler()
scaler.fit(X)                      # learns mean_ and scale_
X_scaled = scaler.transform(X)     # applies them

print("learned mean :", scaler.mean_)
print("learned std  :", scaler.scale_)
print("\nscaled data:\n", X_scaled)

learned mean : [3. 4.]
learned std  : [1.63299316 1.63299316]

scaled data:
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


Manual check for the first entry: $(1-3)/1.6330=-1.2247$, which equals the printed value. The standard deviation $1.6330=\sqrt{8/3}$ uses the population formula (divide by $n$, not $n-1$).

### 3.2 `fit_transform()` and the train/test rule

`fit_transform()` gives the same result as `fit()` followed by `transform()`, in one line.

In [6]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled)
print("column means:", X_scaled.mean(axis=0))
print("column stds :", X_scaled.std(axis=0))

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
column means: [0. 0.]
column stds : [1. 1.]


**The golden rule: fit on training data only.**

* `fit_transform()` → **training** set
* `transform()` → **test** set (re-using the training statistics)

If you call `fit()` again on the test set, the scaler computes *different* statistics, so training and test rows are no longer measured with the same ruler. The experiment below makes this concrete with a test set whose true average is higher than the training average.

In [7]:
rng = np.random.RandomState(0)
X_train = rng.normal(loc=50, scale=10, size=(100, 1))
X_test  = rng.normal(loc=65, scale=10, size=(30, 1))     # distribution has shifted

right = StandardScaler().fit(X_train)                    # correct: statistics from train
wrong = StandardScaler().fit(X_test)                     # wrong  : statistics from test

print(f"train mean / std learned    : {right.mean_[0]:.2f} / {right.scale_[0]:.2f}")
print(f"test mean after CORRECT use : {right.transform(X_test).mean():+.3f}   (shift is still visible)")
print(f"test mean after WRONG use   : {wrong.transform(X_test).mean():+.3f}   (shift has been erased)")

train mean / std learned    : 50.60 / 10.08
test mean after CORRECT use : +1.842   (shift is still visible)
test mean after WRONG use   : +0.000   (shift has been erased)


With the correct procedure the scaled test mean is about **+1.8** standard deviations away from zero, which honestly reflects that the test data drifted. Re-fitting on the test set forces the mean back to 0 and hides the drift. In a real deployment this is exactly the kind of silent error that makes a model look healthier than it is. Pipelines (Section 5) prevent it automatically.

*Preview:* Chapter 2 of the book uses the same three methods with `MinMaxScaler`, `OneHotEncoder` and others.

## 4. Custom estimators and transformers

The API is deliberately **open**: if you subclass the right base classes and respect a few conventions, your own code behaves like a built-in scikit-learn component (it works inside `Pipeline`, `GridSearchCV`, `cross_val_score`, ...).

**Building blocks**

| Class | Gives you |
|-------|-----------|
| `BaseEstimator` | `get_params()` / `set_params()` and a readable `repr`, derived automatically from the `__init__` signature |
| `TransformerMixin` | a ready-made `fit_transform()` |
| `RegressorMixin` | a default `score()` returning $R^2$ |
| `ClassifierMixin` | a default `score()` returning accuracy |

A **mixin** is a small class that contributes one specific ability and is combined with others via multiple inheritance, instead of building one deep inheritance tree.

> **Version note.** In recent releases (≥ 1.6) the mixins must be listed *before* `BaseEstimator`, i.e. `class MyThing(TransformerMixin, BaseEstimator)`. The estimator-tag system (Section 8) relies on that order.

**Conventions to respect**

1. `__init__` only *stores* hyperparameters, with no computation and no modification of the values. This is what lets `get_params`, `set_params` and `clone` work.
2. Things learned from data are stored in attributes that end with an underscore (`lower_`, `n_features_in_`, ...).
3. `fit()` returns `self`.
4. Methods that need a fitted model should call `check_is_fitted()` first.

### 4.1 A custom transformer: `QuantileWinsorizer`

Our transformer learns lower/upper cut-off values from the *training* data (given quantiles) and later squeezes any value outside these bounds back onto them. Because the bounds come from `fit()`, the transformer respects the train/test rule from Section 3.

In [8]:
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from sklearn.utils.validation import check_is_fitted, validate_data


class QuantileWinsorizer(TransformerMixin, BaseEstimator):
    # Cap extreme values at learned quantile bounds (column by column).

    def __init__(self, lower_q=0.05, upper_q=0.95):
        self.lower_q = lower_q                 # store only, never modify
        self.upper_q = upper_q

    def fit(self, X, y=None):
        X = validate_data(self, X)             # checks input, sets n_features_in_
        self.lower_ = np.quantile(X, self.lower_q, axis=0)
        self.upper_ = np.quantile(X, self.upper_q, axis=0)
        return self                            # enables chaining

    def transform(self, X):
        check_is_fitted(self)                  # raises NotFittedError if fit() was skipped
        X = validate_data(self, X, reset=False)
        return np.clip(X, self.lower_, self.upper_)


rng = np.random.RandomState(1)
X_demo = rng.normal(size=(200, 2))
X_demo[0] = [25, -30]                          # two obvious outliers
X_demo[1] = [-40, 18]

wins = QuantileWinsorizer(lower_q=0.01, upper_q=0.99)
X_clean = wins.fit_transform(X_demo)           # fit_transform comes free from TransformerMixin

print("learned lower bounds:", wins.lower_.round(2))
print("learned upper bounds:", wins.upper_.round(2))
print("max |value| before  :", np.abs(X_demo).max().round(2))
print("max |value| after   :", np.abs(X_clean).max().round(2))
print("get_params()        :", wins.get_params())

learned lower bounds: [-2.51 -2.44]
learned upper bounds: [2.53 2.19]
max |value| before  : 40.0
max |value| after   : 2.53
get_params()        : {'lower_q': 0.01, 'upper_q': 0.99}


The extreme values (25, -30, -40, 18) have been pulled back to the 1st/99th percentile bounds, while ordinary values pass through untouched. Notice that `get_params()` worked even though we never wrote it. That comes from `BaseEstimator`.

### 4.2 A custom estimator: `MeanBaselineRegressor`

A model needs `fit()` and `predict()`. The simplest honest model always predicts the training mean. It is not useful on its own, but it is a standard **baseline**: any real model should beat it. By adding `RegressorMixin` we obtain `score()` ($R^2$) without writing it.

In [9]:
class MeanBaselineRegressor(RegressorMixin, BaseEstimator):
    # Always predicts the mean of the training targets.

    def fit(self, X, y):
        X, y = validate_data(self, X, y)
        self.mean_ = float(np.mean(y))
        return self

    def predict(self, X):
        check_is_fitted(self)
        X = validate_data(self, X, reset=False)
        return np.full(X.shape[0], self.mean_)


baseline = MeanBaselineRegressor().fit(X_demo, X_demo[:, 0] * 0.5 + rng.normal(size=200))
print("learned mean_ :", round(baseline.mean_, 4))
print("R^2 on its own training data:", round(baseline.score(X_demo, X_demo[:, 0] * 0.5), 4))

learned mean_ : 0.0924
R^2 on its own training data: -0.003


The $R^2$ of a constant predictor is close to 0 by construction (here slightly negative, $-0.003$; it would be exactly 0 only if scored on the very targets it averaged). That is why it makes a good "floor": any useful model must score clearly above it.

### 4.3 Plugging custom parts into the standard tools

Because both classes follow the conventions, they can be dropped into a `Pipeline` and tuned by `GridSearchCV` like any built-in object. Here we tune the winsorising quantile of our own transformer on a regression problem in which a few rows have corrupted (hugely inflated) feature values.

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.datasets import make_regression

Xr, yr = make_regression(n_samples=300, n_features=4, noise=10, random_state=3)
Xr[:6] *= 30                                     # corrupt a few rows with huge feature values
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.25, random_state=3)

pipe_custom = Pipeline([
    ("winsor", QuantileWinsorizer()),
    ("reg", LinearRegression()),
])

grid = GridSearchCV(
    pipe_custom,
    param_grid={"winsor__lower_q": [0.0, 0.02, 0.05],
                "winsor__upper_q": [1.0, 0.98, 0.95]},
    cv=5,
)
grid.fit(Xr_tr, yr_tr)

print("best params   :", grid.best_params_)
print("CV R^2        : %.4f" % grid.best_score_)
print("test R^2      : %.4f" % grid.score(Xr_te, yr_te))

best params   : {'winsor__lower_q': 0.02, 'winsor__upper_q': 0.95}
CV R^2        : 0.9610
test R^2      : 0.9577


The search treated `winsor__lower_q` and `winsor__upper_q` as ordinary hyperparameters. The `<step name>__<parameter>` (double underscore) syntax is the standard way to address parameters inside pipelines. The book revisits custom components in later chapters.

## 5. Pipelines and workflow automation

### 5.1 Concept

A **pipeline** glues several steps (zero or more transformers, then optionally one final estimator) into a *single estimator*.

* `pipe.fit(X, y)` → calls `fit_transform` on each transformer in order, then `fit` on the last step.
* `pipe.predict(X)` / `pipe.score(X, y)` → calls `transform` on each transformer, then `predict` / `score` on the last step.

Written as a formula, a pipeline with transformers $T_1,\dots,T_K$ and final estimator $E$ computes

$$\hat{\mathbf y}=E\bigl(T_K(\cdots T_2(T_1(\mathbf X)))\bigr).$$

**Why this matters**

* *Correctness*: preprocessing statistics are learned from training data only, so the leakage problem of Section 3 cannot happen by accident.
* *Reproducibility*: the whole recipe is one object.
* *Deployment*: one object can be saved and loaded again, so no separate preprocessing script needs to be kept in sync.
* *Tuning*: the whole recipe can be cross-validated and searched at once.

In [11]:
from sklearn.datasets import make_classification
from sklearn.pipeline import Pipeline, make_pipeline

Xc, yc = make_classification(n_samples=300, n_features=6, n_informative=4, random_state=7)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=7)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipe.fit(Xc_tr, yc_tr)

print("steps          :", [name for name, _ in pipe.steps])
print(f"train accuracy : {pipe.score(Xc_tr, yc_tr):.4f}")
print(f"test accuracy  : {pipe.score(Xc_te, yc_te):.4f}")

steps          : ['scaler', 'clf']
train accuracy : 0.8622
test accuracy  : 0.8800


What happened internally:

| Call | Step 1 (`scaler`) | Step 2 (`clf`) |
|------|-------------------|----------------|
| `pipe.fit(X_train, y_train)` | `fit_transform(X_train)` | `fit(X_train_scaled, y_train)` |
| `pipe.score(X_test, y_test)` | `transform(X_test)`, re-using the **training** mean/std | `score(X_test_scaled, y_test)` |

Training accuracy is about 0.86 and test accuracy about 0.88, so the two are close and there is no sign of over-fitting on this synthetic dataset. The test score is even slightly higher than the training score, which can happen by chance with a small test set (75 samples) and is not a cause for concern.

Two convenient extras:

In [12]:
# 1) make_pipeline: same thing, step names are generated automatically
auto = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xc_tr, yc_tr)
print("auto-generated names:", list(auto.named_steps))

# 2) look inside a fitted step
print("scaler mean_ (first 3 features):", pipe.named_steps["scaler"].mean_[:3].round(3))
print("classifier coef_ shape         :", pipe.named_steps["clf"].coef_.shape)

auto-generated names: ['standardscaler', 'logisticregression']
scaler mean_ (first 3 features): [ 0.191 -0.441  0.314]
classifier coef_ shape         : (1, 6)


### 5.2 Saving the whole workflow

Because the pipeline is one object, persisting it is a one-liner with `joblib` (the standard tool for scikit-learn objects). Loading it later restores *both* the scaler's statistics and the classifier's weights.

In [13]:
import joblib, tempfile, os

path = os.path.join(tempfile.gettempdir(), "demo_pipeline.joblib")
joblib.dump(pipe, path)
restored = joblib.load(path)

same = np.array_equal(pipe.predict(Xc_te), restored.predict(Xc_te))
print("restored pipeline gives identical predictions:", same)

restored pipeline gives identical predictions: True


> **Security reminder:** `joblib`/`pickle` files can execute code when loaded. Only load files that you created yourself or fully trust.

### 5.3 A note on MLOps

The linear workflow above is the first brick of a larger idea. **MLOps** (machine-learning operations) applies software-engineering discipline to the entire life cycle of a model:

* **Continuous training**: re-train when new data arrives or performance drops.
* **CI/CD**: automatically test code and data, and deliver new model versions without manual steps.
* **Monitoring**: watch input/prediction distributions for *drift* and alert when quality degrades.

The book stresses that *no model works well forever*. Data distributions change, so a deployed model needs observation and refreshing. scikit-learn supports this with `Pipeline` (one versionable artefact), `GridSearchCV` (automatic tuning) and `joblib`/`pickle` (persistence), and it integrates with platforms such as MLflow or Kubeflow. Chapter 14 of the book builds a fuller pipeline with `ColumnTransformer`.

## 6. Common attributes and methods

Complex models are hard to interpret, but scikit-learn gives every fitted model a similar "window" through which its learned state can be inspected.

**Naming convention:** anything learned from data ends with an underscore, e.g. `coef_`, `intercept_`, `feature_importances_`, `n_features_in_`. Those attributes **exist only after `fit()`**. Hyperparameters, which you choose yourself, have no trailing underscore.

For a linear model the prediction is

$$\hat y=\mathbf w^{\top}\mathbf x+b=\sum_{j=1}^{p}w_jx_j+b,$$

with `coef_` $=\mathbf w$ and `intercept_` $=b$.

**`score()`** returns a default metric that depends on the model type:

* classifiers → accuracy $=\dfrac{\text{correct predictions}}{\text{all predictions}}$
* regressors → coefficient of determination $R^2=1-\dfrac{\sum_i (y_i-\hat y_i)^2}{\sum_i (y_i-\bar y)^2}$

In [14]:
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

model = LinearRegression().fit(X, y)

print("Coefficients (w):", model.coef_)
print("Intercept   (b):", model.intercept_)
print("R-squared       :", model.score(X, y))

# manual verification of the prediction at x = 3
manual = model.coef_[0] * 3 + model.intercept_
print(f"\nmanual prediction at x=3: {manual:.2f}  (true y = {y[2]})")

Coefficients (w): [0.95]
Intercept   (b): 0.04999999999999938
R-squared       : 0.9809782608695652

manual prediction at x=3: 2.90  (true y = 3.0)


The slope is **0.95**, the intercept is about **0.05** (the tiny floating-point tail `...938` is rounding noise), and $R^2\approx 0.981$, meaning the line explains about 98 % of the variance of $y$.

### 6.1 Why the book warns about plain $R^2$

$R^2$ never decreases when you add another feature, even a useless one, so it can flatter complicated models. The **adjusted** version applies a penalty for the number of predictors $p$ given $n$ samples:

$$R^2_{\text{adj}}=1-(1-R^2)\,\frac{n-1}{n-p-1}.$$

In [15]:
n, p = X.shape
r2 = model.score(X, y)
r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"R^2          = {r2:.4f}")
print(f"adjusted R^2 = {r2_adj:.4f}   (a bit lower: it charges for model complexity)")

R^2          = 0.9810
adjusted R^2 = 0.9746   (a bit lower: it charges for model complexity)


### 6.2 More fitted attributes: a classifier and a feature-name-aware model

* `feature_importances_` is available for tree ensembles.
* `n_features_in_` and `feature_names_in_` record what the model saw during `fit()`. The latter appears when you fit on a `DataFrame`.

In [16]:
import pandas as pd
iris = load_iris(as_frame=True)
Xdf, ydf = iris.data, iris.target

rf = RandomForestClassifier(n_estimators=200, random_state=0).fit(Xdf, ydf)

print("n_features_in_    :", rf.n_features_in_)
print("feature_names_in_ :", list(rf.feature_names_in_))
print("training accuracy :", rf.score(Xdf, ydf))
print("\nfeature importances:")
for name, imp in sorted(zip(rf.feature_names_in_, rf.feature_importances_), key=lambda t: -t[1]):
    print(f"  {name:<20} {imp:.3f}")

n_features_in_    : 4
feature_names_in_ : ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
training accuracy : 1.0

feature importances:
  petal width (cm)     0.437
  petal length (cm)    0.436
  sepal length (cm)    0.098
  sepal width (cm)     0.030


Petal measurements carry most of the importance for separating the three iris species. This is the kind of quick interpretation the shared attributes make possible. Note that the training accuracy is optimistic by nature; for an honest estimate use a held-out set or cross-validation (Section 9, rule 4).

## 7. Hyperparameter tuning with search methods

**Parameters vs. hyperparameters.** Parameters (`coef_`, split thresholds, ...) are *learned* by `fit()`. Hyperparameters (`n_estimators`, `max_depth`, `C`, ...) are *chosen before training* and control how learning happens.

### 7.1 Manual control: `get_params()` and `set_params()`

`set_params()` changes hyperparameters on an existing object and `get_params()` reads all of them. These two methods are also exactly what the automatic search tools use internally.

In [17]:
model = RandomForestClassifier()
model.set_params(n_estimators=100, max_depth=10, random_state=42)

for key, value in sorted(model.get_params().items()):
    print(f"{key:<26} {value}")

bootstrap                  True
ccp_alpha                  0.0
class_weight               None
criterion                  gini
max_depth                  10
max_features               sqrt
max_leaf_nodes             None
max_samples                None
min_impurity_decrease      0.0
min_samples_leaf           1
min_samples_split          2
min_weight_fraction_leaf   0.0
monotonic_cst              None
n_estimators               100
n_jobs                     None
oob_score                  False
random_state               42
verbose                    0
warm_start                 False


A few entries worth understanding:

* **`n_estimators=100`**: number of trees (we set it). More trees reduce variance but cost time.
* **`max_depth=10`**: maximum tree depth (we set it). Limiting depth is a form of regularisation.
* **`max_features='sqrt'`**: default; each split looks at only $\sqrt p$ randomly chosen features, which de-correlates the trees.
* **`criterion='gini'`**: default split-quality measure, the Gini impurity $G=1-\sum_c p_c^2$ (0 for a pure node, 0.5 for a 50/50 binary node).
* **`random_state=42`**: fixes the random number generator so that results are reproducible.

### 7.2 Automatic search

| Tool | Strategy | Good when |
|------|----------|-----------|
| `GridSearchCV` | tries **every** combination in a grid | few hyperparameters, small value lists |
| `RandomizedSearchCV` | samples a fixed number of random combinations (can draw from continuous distributions) | many hyperparameters or wide ranges |
| `HalvingGridSearchCV` / `HalvingRandomSearchCV` | *successive halving*: start all candidates on a little data, keep the best, give survivors more data | large search spaces and expensive models |

All of them evaluate candidates with **cross-validation** and, by default, **refit** the best configuration on the whole training set (`best_estimator_`).

In [18]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

pipe = Pipeline([("scaler", StandardScaler()),
                 ("clf", LogisticRegression(max_iter=1000))])

# --- Grid search -------------------------------------------------
grid = GridSearchCV(pipe, param_grid={"clf__C": [0.01, 0.1, 1, 10]}, cv=5)
grid.fit(Xc_tr, yc_tr)

print("GRID   best params :", grid.best_params_)
print(f"GRID   CV accuracy : {grid.best_score_:.4f}")
print(f"GRID   test accuracy: {grid.score(Xc_te, yc_te):.4f}")

GRID   best params : {'clf__C': 1}
GRID   CV accuracy : 0.8578
GRID   test accuracy: 0.8800


In [19]:
from scipy.stats import loguniform, randint

# --- Randomized search -------------------------------------------
rf_pipe = Pipeline([("clf", RandomForestClassifier(random_state=0))])

rand = RandomizedSearchCV(
    rf_pipe,
    param_distributions={
        "clf__n_estimators": randint(50, 300),
        "clf__max_depth": randint(2, 12),
        "clf__min_samples_leaf": randint(1, 8),
    },
    n_iter=15, cv=5, random_state=0,
)
rand.fit(Xc_tr, yc_tr)

print("RANDOM best params  :", rand.best_params_)
print(f"RANDOM CV accuracy  : {rand.best_score_:.4f}")
print(f"RANDOM test accuracy: {rand.score(Xc_te, yc_te):.4f}")

RANDOM best params  : {'clf__max_depth': 9, 'clf__min_samples_leaf': 1, 'clf__n_estimators': 243}
RANDOM CV accuracy  : 0.8889
RANDOM test accuracy: 0.8533


In [20]:
# --- Successive halving (still behind an 'experimental' import) ---
from sklearn.experimental import enable_halving_search_cv   # noqa: F401
from sklearn.model_selection import HalvingGridSearchCV

halv = HalvingGridSearchCV(
    RandomForestClassifier(random_state=0),
    param_grid={"n_estimators": [50, 150, 300], "max_depth": [3, 6, None]},
    cv=3, factor=3, random_state=0,
)
halv.fit(Xc_tr, yc_tr)

print("HALVING best params :", halv.best_params_)
print(f"HALVING test accuracy: {halv.score(Xc_te, yc_te):.4f}")
print("candidates per round :", halv.n_candidates_)

HALVING best params : {'max_depth': None, 'n_estimators': 300}
HALVING test accuracy: 0.8533
candidates per round : [9, 3, 1]


The `clf__C` notation again addresses a hyperparameter *inside* a pipeline step. The tuning is performed **inside** cross-validation, so for each fold the scaler is re-fitted on that fold's training part only, with no leakage. Always report the final score on a test set the search never touched (as done above).

Halving shows its strength in `n_candidates_`: the number of surviving candidates shrinks every round, so weak settings are discarded cheaply.

## 8. Working with metadata: tags and routing

scikit-learn uses two kinds of "data about the work":

### 8.1 Estimator tags

Each estimator carries **tags** that describe its capabilities (does it accept missing values? sparse input? is it a classifier or regressor? does it need `y`?). Tools such as pipelines and search objects can read the tags to decide how to treat an estimator instead of relying on trial and error.

In [21]:
from sklearn.ensemble import HistGradientBoostingClassifier

rows = []
for est in [LogisticRegression(), HistGradientBoostingClassifier(), LinearRegression()]:
    t = est.__sklearn_tags__()
    rows.append({
        "estimator": type(est).__name__,
        "estimator_type": t.estimator_type,
        "accepts NaN": t.input_tags.allow_nan,
        "accepts sparse": t.input_tags.sparse,
        "needs y": t.target_tags.required,
    })

pd.DataFrame(rows).set_index("estimator")

,estimator_type,accepts NaN,accepts sparse,needs y
estimator,,,,
LogisticRegression,classifier,False,True,True
HistGradientBoostingClassifier,classifier,True,False,True
LinearRegression,regressor,False,True,True


The table shows a real difference: `HistGradientBoostingClassifier` natively accepts missing values (`allow_nan=True`) but does not take sparse matrices, while the two linear models behave the opposite way. When you write your own estimator (Section 4) you can override tags so that other tools know what it supports.

### 8.2 Metadata routing

Sometimes extra information must travel *alongside* `X` and `y`: sample weights, group labels, etc. **Metadata routing** controls where such information goes in a composite object:

* **Consumers** use the metadata (e.g. a classifier that supports `sample_weight`).
* **Routers** (e.g. `Pipeline`, `GridSearchCV`) receive the metadata and forward it to the right consumers.

Each consumer declares what it wants with `set_fit_request(sample_weight=True/False)` (similar methods exist for `score`, `predict`, ...). The feature is opt-in via a configuration flag.

*Motivating case:* class-imbalanced data. We want the classifier to up-weight the minority class, but we do **not** want the scaler to use those weights.

In [22]:
import sklearn

# imbalanced-looking weights: minority class (label 1) gets 3x the weight
weights = np.where(yc_tr == 1, 3.0, 1.0)

with sklearn.config_context(enable_metadata_routing=True):

    # StandardScaler.fit can also accept sample_weight -> we must say explicitly we do NOT want it
    scaler = StandardScaler().set_fit_request(sample_weight=False)
    # the classifier is the consumer that should receive the weights
    clf = LogisticRegression(max_iter=1000).set_fit_request(sample_weight=True)

    routed = Pipeline([("scaler", scaler), ("clf", clf)])
    routed.fit(Xc_tr, yc_tr, sample_weight=weights)     # the weights reach only 'clf'

    print("weighted pipeline test accuracy:", round(routed.score(Xc_te, yc_te), 4))

weighted pipeline test accuracy: 0.8267


What if the request is **not** declared? With routing enabled, scikit-learn refuses to guess and raises an informative error, which is the safety net that stops weights from being silently ignored or silently misused.

In [23]:
with sklearn.config_context(enable_metadata_routing=True):
    undeclared = Pipeline([("scaler", StandardScaler()),
                           ("clf", LogisticRegression(max_iter=1000))])
    try:
        undeclared.fit(Xc_tr, yc_tr, sample_weight=weights)
    except Exception as e:
        print(type(e).__name__)
        print(str(e)[:260], "...")

UnsetMetadataPassedError
[sample_weight] are passed but are not explicitly set as requested or not requested for StandardScaler.fit_transform, which is used within Pipeline.fit. Call `StandardScaler.set_fit_request({metadata}=True/False).set_transform_request({metadata}=True/False)` f ...


## 9. Best practices for API usage

1. **Rely on the uniform API.** Stick to `fit` / `predict` / `transform` so that models are interchangeable and code stays readable.
2. **Pre-process before modelling.** Scale, encode and impute using `sklearn.preprocessing` / `sklearn.impute` (Chapters 2-3).
3. **Use pipelines** for any workflow with more than one step: less leakage, one deployable object, simpler tuning.
4. **Validate with cross-validation** (`cross_val_score`, or the CV inside `GridSearchCV`) instead of trusting a single split or the training score.
5. **Tune hyperparameters systematically**: grid search for small spaces, randomized or halving search for large ones, and defaults only as a starting point.
6. **Inspect what the model learned** via `coef_`, `feature_importances_`, `score()` and friends.
7. **Keep components modular**, with small custom transformers and one concern per step, so debugging and scaling stay manageable.

A compact demonstration that combines rules 1 to 5 in a few lines:

In [24]:
from sklearn.model_selection import cross_val_score

best = grid.best_estimator_                     # tuned pipeline from Section 7
scores = cross_val_score(best, Xc, yc, cv=5)    # honest estimate on the full data

print("5-fold accuracies:", scores.round(3))
print(f"mean ± std       : {scores.mean():.3f} ± {scores.std():.3f}")

5-fold accuracies: [0.817 0.883 0.883 0.833 0.883]
mean ± std       : 0.860 ± 0.029


## Chapter summary

| Concept | Key methods / objects | Role |
|---------|----------------------|------|
| **Estimator** | `fit()`, `predict()`, `fit_predict()` | learn from data, make predictions |
| **Transformer** | `fit()`, `transform()`, `fit_transform()` | change or prepare data |
| **Custom components** | `BaseEstimator`, `TransformerMixin`, `RegressorMixin`, `ClassifierMixin` | extend the library with your own, fully compatible classes |
| **Pipeline** | `Pipeline`, `make_pipeline`, `named_steps` | chain steps into a single estimator |
| **Learned attributes** | `coef_`, `intercept_`, `feature_importances_` (trailing `_`) | inspect what was learned |
| **Evaluation** | `score()` | default quick metric ($R^2$ or accuracy) |
| **Hyperparameters** | `get_params()`, `set_params()`, `GridSearchCV`, `RandomizedSearchCV`, `HalvingGridSearchCV` | configure and tune |
| **Metadata** | estimator tags, `set_fit_request()`, metadata routing | describe capabilities, route extra information |

### Key takeaways

* The whole library is built on **one small vocabulary**; learning it once pays off in every later chapter.
* **Fit on training data, transform everything else.** Pipelines enforce this automatically.
* **Custom code is a first-class citizen** if it follows the conventions (store hyperparameters only, learned attributes end with `_`, `fit` returns `self`).
* **Tuning belongs inside cross-validation**, and the final score belongs on data the search never saw.

### Self-check questions

<details><summary><b>Q1.</b> Why do <code>fit()</code> and <code>fit_predict()</code> both exist?</summary>

`fit()` learns; `predict()` applies the model to (usually new) data. `fit_predict()` is a shortcut for the case where predictions on the *training data itself* are the goal, typically clustering.
</details>

<details><summary><b>Q2.</b> What goes wrong if you call <code>fit()</code> on the test set of a scaler?</summary>

The scaler learns test-set statistics, so train and test are put on different scales and distribution shift is hidden (see the experiment in Section 3.2). It is a form of data leakage.
</details>

<details><summary><b>Q3.</b> Why must <code>__init__</code> of a custom estimator not do any computation?</summary>

<code>get_params</code>, <code>set_params</code> and <code>clone</code> rebuild objects from the constructor arguments; if <code>__init__</code> altered or computed values, searches and cross-validation would behave inconsistently.
</details>

<details><summary><b>Q4.</b> What does the trailing underscore in <code>coef_</code> signify?</summary>

The attribute was learned from data and exists only after <code>fit()</code>.
</details>

<details><summary><b>Q5.</b> When would you pick <code>RandomizedSearchCV</code> over <code>GridSearchCV</code>?</summary>

When there are many hyperparameters or continuous ranges, so that the full grid would be too large; random sampling covers the space with a fixed budget.
</details>

---
**Reference:** Sukup, J. *scikit-learn Cookbook, Third Edition*, Chapter 1 — "Common Conventions and API Elements of scikit-learn". Examples were re-run and extended by the author of this notebook; explanations are paraphrased and supplemented with additional experiments.